# HR RAG Chatbot - Jupyter Notebook
## Building a Retrieval-Augmented Generation System for HR Documents

This notebook demonstrates building an intelligent HR assistant using LangChain, FAISS vector stores, and Google Gemini API.

## Section 1: Import Required Libraries

In [ ]:
import os
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

print("✓ All libraries imported successfully")

## Section 2: Configure API Keys

In [ ]:
# Set your Google Gemini API key
os.environ["GOOGLE_API_KEY"] = "your-google-api-key-here"

print("✓ API keys configured")

## Section 3: Document Loading

In [ ]:
# Load HR documents from a directory
pdf_loader = DirectoryLoader(
    "./hr_documents",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader
)

docx_loader = DirectoryLoader(
    "./hr_documents",
    glob="**/*.docx",
    loader_cls=Docx2txtLoader
)

pdf_docs = pdf_loader.load()
docx_docs = docx_loader.load()
all_documents = pdf_docs + docx_docs

print(f"✓ Loaded {len(all_documents)} documents")

## Section 4: Text Splitting

In [ ]:
# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(all_documents)

print(f"✓ Split into {len(chunks)} chunks")

## Section 5: Embeddings & Vector Store

In [ ]:
# Create embeddings and store in FAISS
embeddings = SentenceTransformerEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("✓ Vector store created")

## Section 6: Build RAG Chain with Chat History

In [ ]:
# Initialize Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-pro",
    temperature=0.3
)

# Define prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful HR assistant. Answer questions based only 
    on the provided HR documents context. If unsure, say so.
    
    Context: {context}"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

# Build RAG chain
rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough(),
        "chat_history": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# Add chat history support
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain_with_history = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history"
)

print("✓ RAG chain with history built")

## Section 7: Run the Chatbot

In [ ]:
# Chat with the HR assistant
session_id = "hr_session_1"

def chat(question):
    response = chain_with_history.invoke(
        {"question": question},
        config={"configurable": {"session_id": session_id}}
    )
    print(f"You: {question}")
    print(f"HR Bot: {response}")
    print("-" * 50)
    return response

# Example questions
chat("What is the leave policy?")
chat("How many sick days do employees get?")